**Load Dataset from Kaggle to Colab**

In [4]:
# %pip install Kagglehub

In [5]:
# import os
# os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_KEY

In [6]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("shubhammehta21/movie-lens-small-latest-dataset")

# print("Path to dataset files:", path)

Make a folder Dataset and copy the path

In [7]:
# import shutil
# # copy dataset from source_dir to dest_dir
# shutil.copytree(
#     "/kaggle/input/movie-lens-small-latest-dataset",
#     "/content/Dataset",
#     dirs_exist_ok=True,
# )

In [8]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


**Load dataset**

In [11]:
movies = pd.read_csv(r"D:\My Learning\ML projects\Movie-Recommendation-System\datasets\movies.csv")
ratings = pd.read_csv(r"D:\My Learning\ML projects\Movie-Recommendation-System\datasets\ratings.csv")
links = pd.read_csv(r"D:\My Learning\ML projects\Movie-Recommendation-System\datasets\links.csv")

**EDA**

In [12]:
print("Shape:", movies.shape)

Shape: (9742, 3)


In [13]:
print("Shape:", ratings.shape)

Shape: (100836, 4)


In [14]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [15]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [16]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [ ]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [ ]:
links.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


In [ ]:
movies['genres'] = movies['genres'].fillna('')

In [ ]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
movies['genres'].shape

(9742,)

In [ ]:
print(movies['genres'])

0       Adventure|Animation|Children|Comedy|Fantasy
1                        Adventure|Children|Fantasy
2                                    Comedy|Romance
3                              Comedy|Drama|Romance
4                                            Comedy
                           ...                     
9737                Action|Animation|Comedy|Fantasy
9738                       Animation|Comedy|Fantasy
9739                                          Drama
9740                               Action|Animation
9741                                         Comedy
Name: genres, Length: 9742, dtype: object


**Pre Processing**

In [ ]:
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)
print(movies['genres'])

0       Adventure Animation Children Comedy Fantasy
1                        Adventure Children Fantasy
2                                    Comedy Romance
3                              Comedy Drama Romance
4                                            Comedy
                           ...                     
9737                Action Animation Comedy Fantasy
9738                       Animation Comedy Fantasy
9739                                          Drama
9740                               Action Animation
9741                                         Comedy
Name: genres, Length: 9742, dtype: object


In [ ]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
# Compute average rating upto one decimal place and reset index
average_ratings = ratings.groupby('movieId')['rating'].mean().reset_index().round(1)

# Optional: Rename column for clarity
average_ratings.rename(columns={'rating': 'avg_rating'}, inplace=True)

# print(average_ratings)
average_ratings.head()

,movieId,avg_rating
0,1,3.9
1,2,3.4
2,3,3.3
3,4,2.4
4,5,3.1


In [ ]:
# Merge on 'movieId' to add the 'avg_rating' column
# 'how="left"' ensures you keep all rows from your original movies dataframe
movies = pd.merge(movies, average_ratings[['movieId', 'avg_rating']], on='movieId', how='left')

movies.head()

,movieId,title,genres,avg_rating
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,3.9
1,2,Jumanji (1995),Adventure Children Fantasy,3.4
2,3,Grumpier Old Men (1995),Comedy Romance,3.3
3,4,Waiting to Exhale (1995),Comedy Drama Romance,2.4
4,5,Father of the Bride Part II (1995),Comedy,3.1


In [ ]:
# Extract movie name and year into separate columns
movies[['title', 'year']] = movies['title'].str.extract(r'^(.*?)\s*\((\d{4})\)$')

# Clean up whitespace if needed
movies['title'] = movies['title'].str.strip()

movies.head()

,movieId,title,genres,avg_rating,year
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,3.9,1995
1,2,Jumanji,Adventure Children Fantasy,3.4,1995
2,3,Grumpier Old Men,Comedy Romance,3.3,1995
3,4,Waiting to Exhale,Comedy Drama Romance,2.4,1995
4,5,Father of the Bride Part II,Comedy,3.1,1995


**Compute Similarity Scores**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer


In [ ]:
# Convert text to vectors
vectorizer = CountVectorizer()
vector_matrix = vectorizer.fit_transform(movies['genres'])
# print(vector_matrix)

similarity_matrix = cosine_similarity(vector_matrix)

# Format the output into a readable DataFrame (Optional)
df_similarity = pd.DataFrame(
    similarity_matrix,
    columns=[f"Doc {i+1}" for i in range(len(movies['genres']))],
    index=[f"Doc {i+1}" for i in range(len(movies['genres']))]
)

# print("--- Cosine Similarity Matrix ---")
# print(df_similarity)

In [ ]:
# Check the type
if isinstance(similarity_matrix , pd.DataFrame):
    print("The variable is a DataFrame!")
else:
    print("The variable is NOT a DataFrame.")

The variable is NOT a DataFrame.


In [ ]:
def get_movie_recommendations(idx, cosine_sim_matrix, df, top_n=5):
    """
    Recommends top_n movies similar to the given movie index based on a cosine similarity matrix.

    Parameters:
    idx (int): The index of the movie you want recommendations for.
    cosine_sim_matrix (np.ndarray or pd.DataFrame): The precomputed cosine similarity matrix.
    df (pd.DataFrame): DataFrame containing the 'title' column.
    top_n (int): Number of recommendations to return.

    Returns:
    list: Titles of the recommended movies.
    """

    # Get the pairwise similarity scores for all movies with this movie
    # Note: If cosine_sim_matrix is a DataFrame, use .iloc[idx]
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar movies
    # (Skip the first one since it's the movie itself, which has a similarity of 1.0)
    top_indices = [i[0] for i in sim_scores[1:top_n + 1]]

    return  top_indices


In [ ]:
movies.head(10)

,movieId,title,genres,avg_rating,year
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,3.9,1995
1,2,Jumanji,Adventure Children Fantasy,3.4,1995
2,3,Grumpier Old Men,Comedy Romance,3.3,1995
3,4,Waiting to Exhale,Comedy Drama Romance,2.4,1995
4,5,Father of the Bride Part II,Comedy,3.1,1995
5,6,Heat,Action Crime Thriller,3.9,1995
6,7,Sabrina,Comedy Romance,3.2,1995
7,8,Tom and Huck,Adventure Children,2.9,1995
8,9,Sudden Death,Action,3.1,1995
9,10,GoldenEye,Action Adventure Thriller,3.5,1995


# **Predictions**

In [ ]:
movie_name = "Sudden Death"
# 1. Check if the movie exists in the dataframe
matching_movies = movies[movies['title'].str.lower() == movie_name.lower()]
if matching_movies.empty:
    print(f"Error: '{movie_name}' not found in the dataset.")
else:
    # Get the index of the movie
    idx = matching_movies.index[0]
    movie_genre = movies['genres'].iloc[idx]
    movie_year = movies['year'].iloc[idx]
    movie_avg_rating = movies['avg_rating'].iloc[idx]

    # print(f"Movie: {movie_name}")
    # print(f"Genre: {movie_genre}")
    # print(f"Year: {movie_year}")
    # print(f"Average Rating: {movie_avg_rating}")

    indices = get_movie_recommendations(idx, similarity_matrix, movies, top_n=10)
    recommendations_titles = movies['title'].iloc[indices].tolist()
    recommendations_genres = movies['genres'].iloc[indices].tolist()
    recommendations_avg_ratings = movies['avg_rating'].iloc[indices].tolist()
    recommendations_release_date = movies['year'].iloc[indices].tolist()
    # print("Recommended movies:")
    print(recommendations_titles)
    print(recommendations_genres)
    print(recommendations_avg_ratings)
    # print(recommendations_avg_ratings)


['Fair Game', 'Under Siege 2: Dark Territory', 'Hunted, The', 'Bloodsport 2 (a.k.a. Bloodsport II: The Next Kumite)', 'Best of the Best 3: No Turning Back', 'Double Team', 'Steel', 'Knock Off', 'Avalanche', 'Aces: Iron Eagle III']
['Action', 'Action', 'Action', 'Action', 'Action', 'Action', 'Action', 'Action', 'Action', 'Action']
[1.7, 2.7, 3.0, 2.8, 2.5, 4.0, 2.3, 5.0, 2.5, 2.0]
